# Caltech-101 Image Classification: Data Loading and Preprocessing

This notebook handles data loading, exploration, and preprocessing for the Caltech-101 dataset.
It prepares the data for training classical ML and deep learning models.

## Section 1: Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shutil
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")
print(f"Working directory: {os.getcwd()}")

## Section 2: Load and Explore Caltech-101 Dataset

In [ ]:
# Define the path to Caltech-101 dataset
dataset_path = Path("e:/jingxizhang/caltech-101")

# List all categories (subdirectories)
categories = sorted([d for d in dataset_path.iterdir() if d.is_dir() and not d.name.startswith('.')])
category_names = [c.name for c in categories]

print(f"Total number of categories: {len(category_names)}")
print(f"Categories: {category_names[:10]}...") # Print first 10

# Collect all images and their labels
image_paths = []
labels = []
image_counts = {}

for category_id, category_dir in enumerate(categories):
    image_files = list(category_dir.glob("*.jpg")) + list(category_dir.glob("*.png"))
    image_counts[category_dir.name] = len(image_files)
    
    for img_file in image_files:
        image_paths.append(str(img_file))
        labels.append(category_id)

print(f"\nTotal images found: {len(image_paths)}")
print(f"Label distribution:")
label_df = pd.Series(image_counts).sort_values(ascending=False)
print(label_df)

# Create a dataframe with image paths and labels
data_df = pd.DataFrame({
    'image_path': image_paths,
    'label': labels,
    'category_name': [category_names[l] for l in labels]
})

print(f"\nDataset shape: {data_df.shape}")
print(data_df.head())

In [ ]:
# Visualize dataset statistics
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Images per category
label_df.head(20).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 Categories by Image Count')
axes[0].set_xlabel('Number of Images')
axes[0].grid(axis='x', alpha=0.3)

# Plot 2: Distribution histogram
axes[1].hist(list(image_counts.values()), bins=20, color='steelblue', edgecolor='black')
axes[1].set_title('Distribution of Images per Category')
axes[1].set_xlabel('Number of Images')
axes[1].set_ylabel('Number of Categories')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/01_dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Dataset statistics visualization saved!")

In [ ]:
# Visualize sample images from different categories
fig, axes = plt.subplots(5, 5, figsize=(15, 15))
axes = axes.flatten()

# Sample one image per category (up to 25)
np.random.seed(42)
for idx, category_dir in enumerate(categories[:25]):
    image_files = list(category_dir.glob("*.jpg")) + list(category_dir.glob("*.png"))
    if image_files:
        sample_img = Image.open(np.random.choice(image_files))
        axes[idx].imshow(sample_img)
        axes[idx].set_title(category_dir.name, fontsize=8)
        axes[idx].axis('off')

plt.suptitle('Sample Images from Different Categories (Caltech-101)', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/01_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

print("Sample images visualization saved!")

## Section 3: Data Preprocessing and Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# Create train/val/test split (70%, 15%, 15%)
# First split: 70% train, 30% temp (val+test)
train_idx, temp_idx = train_test_split(
    range(len(data_df)), 
    test_size=0.30, 
    random_state=42, 
    stratify=data_df['label']
)

# Second split: Split temp into 50% val and 50% test (15% each)
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.50, 
    random_state=42, 
    stratify=data_df.iloc[temp_idx]['label']
)

# Create split labels
data_df['split'] = 'train'
data_df.loc[data_df.index.isin(val_idx), 'split'] = 'val'
data_df.loc[data_df.index.isin(test_idx), 'split'] = 'test'

print("Data split distribution:")
print(data_df['split'].value_counts())
print(f"\nTrain: {len(train_idx)} ({len(train_idx)/len(data_df)*100:.1f}%)")
print(f"Val: {len(val_idx)} ({len(val_idx)/len(data_df)*100:.1f}%)")
print(f"Test: {len(test_idx)} ({len(test_idx)/len(data_df)*100:.1f}%)")

# Save the split information
data_df.to_csv('e:/jingxizhang/image-classification-project/results/data_split.csv', index=False)
print("\nData split information saved!")

In [ ]:
# Function to load and preprocess images
def load_image_rgb(image_path, target_size=(256, 256)):
    """Load image and convert to RGB"""
    try:
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img = img.resize(target_size, Image.LANCZOS)
        return np.array(img)
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None

# Preprocess a subset to verify
print("Preprocessing sample images...")
sample_images = []
valid_paths = []

for idx in range(min(100, len(data_df))):
    img = load_image_rgb(data_df.iloc[idx]['image_path'])
    if img is not None:
        sample_images.append(img)
        valid_paths.append(data_df.iloc[idx]['image_path'])

print(f"Successfully loaded {len(sample_images)} images")
print(f"Image shape: {sample_images[0].shape}")
print(f"Image dtype: {sample_images[0].dtype}")
print(f"Value ranges: {sample_images[0].min()} to {sample_images[0].max()}")

In [ ]:
# Visualize preprocessing effect
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for idx in range(5):
    # Original image from file
    img_path = data_df.iloc[idx]['image_path']
    original = Image.open(img_path)
    axes[0, idx].imshow(original)
    axes[0, idx].set_title(f'Original\n{original.size}', fontsize=8)
    axes[0, idx].axis('off')
    
    # Preprocessed image
    preprocessed = load_image_rgb(img_path)
    axes[1, idx].imshow(preprocessed)
    axes[1, idx].set_title(f'Preprocessed\n256x256', fontsize=8)
    axes[1, idx].axis('off')

plt.suptitle('Image Preprocessing: Original vs Resized', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/01_preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Preprocessing visualization saved!")

In [ ]:
# Data augmentation functions
from torchvision import transforms

# Define augmentation pipelines
train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.RandomAutocontrast(p=0.3),
])

val_test_transform = transforms.Compose([
    # No augmentation for val/test
])

print("Data augmentation pipelines created!")
print("\nTrain augmentation transformations:")
print("- Horizontal flip (50%)")
print("- Rotation (±15 degrees)")
print("- Translation (±10%)")
print("- Color jitter (brightness, contrast, saturation)")
print("- Perspective transform")
print("- Auto contrast (30%)")

In [ ]:
# Visualize data augmentation effects
from PIL import Image as PILImage
import torchvision.transforms.functional as F

sample_img_path = data_df.iloc[10]['image_path']
sample_pil = PILImage.open(sample_img_path).resize((224, 224))

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

# Original
axes[0].imshow(sample_pil)
axes[0].set_title('Original Image', fontsize=10)
axes[0].axis('off')

# Apply various augmentations
np.random.seed(42)
for i in range(1, 10):
    augmented = train_augmentation(sample_pil)
    axes[i].imshow(augmented)
    axes[i].set_title(f'Augmentation {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('e:/jingxizhang/image-classification-project/figures/01_data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Data augmentation visualization saved!")

In [ ]:
# Create summary statistics
summary_stats = {
    'Total Images': len(data_df),
    'Number of Classes': len(category_names),
    'Train Set Size': len(train_idx),
    'Validation Set Size': len(val_idx),
    'Test Set Size': len(test_idx),
    'Image Size (after resize)': '256x256',
    'Number of Channels': 3,
    'Augmentation Techniques': 6
}

summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Property', 'Value'])
print("\n" + "="*50)
print("DATA PREPROCESSING SUMMARY")
print("="*50)
print(summary_df.to_string(index=False))
print("="*50)

# Save summary
summary_df.to_csv('e:/jingxizhang/image-classification-project/results/preprocessing_summary.csv', index=False)
print("\nPreprocessing summary saved!")